## Text input

https://platform.openai.com/docs/models

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
from langchain.agents import create_agent
from langchain_groq import ChatGroq

groq_model = ChatGroq(model="llama-3.1-8b-instant")  # use whichever model worked for you

agent = create_agent(
    model=groq_model,
    system_prompt="You are a science fiction writer, create a capital city at the users request.",
)

In [4]:
from langchain.messages import HumanMessage

question = HumanMessage(content=[
    {"type": "text", "text": "What is the capital of The Moon?"}
])

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

You want a capital for the Moon.  Let me create a breathtaking, futuristic city for you.

**Welcome to Lunarhaven**

Located in the vast, barren expanse of the Moon's southern pole, Lunarhaven is a marvel of engineering and design. This thriving metropolis serves as the capital of the Moon, a hub for interplanetary trade, innovation, and discovery.

**Architectural Marvel**

Lunarhaven's design is inspired by the natural beauty of the Moon's craters and the eerie, otherworldly landscapes that cover its surface. The city's foundation is built into the wall of a massive impact crater, with towering spires and curved silhouettes that seem to defy gravity. The buildings are constructed from a unique blend of locally sourced lunar regolith and advanced, lightweight materials, ensuring a sustainable and eco-friendly environment.

**Districts**

Lunarhaven is divided into six distinct districts, each serving a specific purpose:

1. **The Spire of the Moon**: The central hub of the city, featu

## Image input

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "ipywidgets"], check=False)

CompletedProcess(args=['c:\\Users\\ramiz\\AppData\\Local\\Python\\pythoncore-3.14-64\\python.exe', '-m', 'pip', 'install', 'ipywidgets'], returncode=0)

In [ ]:
# Cell 1 - imports
import base64
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from dotenv import load_dotenv
import os
load_dotenv()

True

In [1]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

In [2]:
print(uploader.value)

({'name': 'capital.jpg', 'type': 'image/jpeg', 'size': 8052, 'content': <memory at 0x000001FC036B8100>, 'last_modified': datetime.datetime(2026, 6, 13, 10, 1, 6, 181000, tzinfo=datetime.timezone.utc)},)


In [13]:
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [14]:
import requests, os
from dotenv import load_dotenv
load_dotenv()

response = requests.get(
    "https://openrouter.ai/api/v1/models",
    headers={"Authorization": f"Bearer {os.getenv('OPENROUTER_API_KEY')}"}
)

models = response.json()["data"]

# Filter free vision models only
free_vision = [
    m["id"] for m in models 
    if str(m.get("pricing", {}).get("prompt", "1")) == "0"
    and "image" in str(m.get("architecture", {}).get("input_modalities", ""))
]

print(f"Found {len(free_vision)} free vision models:")
for m in free_vision:
    print(" -", m)

Found 9 free vision models:
 - nex-agi/nex-n2-pro:free
 - nvidia/nemotron-3.5-content-safety:free
 - nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free
 - google/gemma-4-26b-a4b-it:free
 - google/gemma-4-31b-it:free
 - google/lyria-3-pro-preview
 - google/lyria-3-clip-preview
 - openrouter/free
 - nvidia/nemotron-nano-12b-v2-vl:free


In [ ]:

# Nemotron-nano-12b-v2-vl by NVIDIA (free vision)
openrouter_model = ChatOpenAI(
    model="nvidia/nemotron-nano-12b-v2-vl:free",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)
agent = create_agent(model=openrouter_model, tools=[])

multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this image"},
    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{img_b64}"}}
])

response = agent.invoke({"messages": [multimodal_question]})
print(response['messages'][-1].content)

The image is a picturesque symbol of the Mughal Empire in India, particularly during the rule of Shah Jahan, who commissioned this as his wife Mumaz's mausoleum. The Taj Mahal symbolizes the architectural splendor of the Mughal era. The Taj Mahal is not just the perfect example of Mughal architecture, but it also stands as the perfect example of romantic heritage of that period. Let us explore the inside of the Taj Mahal.



## Audio input

In [20]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "sounddevice", "scipy", "tqdm"], check=False)

CompletedProcess(args=['c:\\Users\\ramiz\\AppData\\Local\\Python\\pythoncore-3.14-64\\python.exe', '-m', 'pip', 'install', 'sounddevice', 'scipy', 'tqdm'], returncode=0)

In [56]:
# Run this and SPEAK into your mic during the 5 seconds!
import sounddevice as sd
from scipy.io.wavfile import write
import base64, io, time, requests, os
from tqdm import tqdm
from dotenv import load_dotenv
load_dotenv()

duration = 5
sample_rate = 44100

print("🎙️ Recording... SPEAK NOW!")
audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1)
for _ in tqdm(range(duration * 10)):
    time.sleep(0.1)
sd.wait()
print("✅ Done recording!")

buf = io.BytesIO()
write(buf, sample_rate, audio)
wav_bytes = buf.getvalue()

# Transcribe
transcription = requests.post(
    "https://api.groq.com/openai/v1/audio/transcriptions",
    headers={"Authorization": f"Bearer {os.getenv('GROQ_API_KEY')}"},
    files={"file": ("audio.wav", wav_bytes, "audio/wav")},
    data={"model": "whisper-large-v3"}
)
transcript_text = transcription.json()["text"]
print("📝 You said:", transcript_text)

🎙️ Recording... SPEAK NOW!


100%|██████████| 50/50 [00:05<00:00,  9.81it/s]


✅ Done recording!
📝 You said:  Tell me a joke.


In [57]:
# ============ RESPOND with Groq LLM (same invoke pattern as original) ============
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

groq_model = ChatGroq(model="llama-3.1-8b-instant")
agent = create_agent(model=groq_model, tools=[])

multimodal_question = HumanMessage(content=f"The user said: '{transcript_text}'. Respond helpfully.")

response = agent.invoke({"messages": [multimodal_question]})
print(response['messages'][-1].content)

A man walked into a library and asked the librarian, "Do you have any books on Pavlov's dogs and Schrödinger's cat?" 

The librarian replied, "It rings a bell, but I'm not sure if it's here or not."
